In [ ]:
# -------------------------------------------------------------
# PART 3: Tasks 3.5 & 3.6 Breaking Residuals & Normalization Layers
# -------------------------------------------------------------
print("--- Testing Network Path Deletions ---")

# 1. Aggressive FFN Reduction
ffn_broken_config = TransformerConfig(vocabulary_size=50257, context_window=256, num_layers=2, num_heads=2, embedding_dim=128, ffn_multiplier=1)
ffn_broken_model = NanoTransformer(ffn_broken_config).to(DEVICE)

# 2. No Residual Connections
no_res_config = TransformerConfig(vocabulary_size=50257, context_window=256, num_layers=2, num_heads=2, embedding_dim=128, use_residual_connections=False)
no_res_model = NanoTransformer(no_res_config).to(DEVICE)

# 3. No Layer Normalization
no_ln_config = TransformerConfig(vocabulary_size=50257, context_window=256, num_layers=2, num_heads=2, embedding_dim=128, use_layernorm=False)
no_ln_model = NanoTransformer(no_ln_config).to(DEVICE)

print("\nEvaluating Baseline vs. Ablated Configurations (100 steps)...")
h_ffn = train_model(ffn_broken_model, "FFN Reduced", training_steps=100, evaluation_interval=100)
h_res = train_model(no_res_model, "No Residuals", training_steps=100, evaluation_interval=100)
h_ln = train_model(no_ln_model, "No LayerNorm", training_steps=100, evaluation_interval=100)

print("\n--- Final Ablation Comparative Loss Table ---")
print(f"Aggressive FFN Reduction Val Loss  : {h_ffn['validation_loss'][-1]:.4f}")
print(f"No Residual Connections Val Loss  : {h_res['validation_loss'][-1]:.4f} (Expect exploding values/stagnation)")
print(f"No Layer Normalization Val Loss   : {h_ln['validation_loss'][-1]:.4f}")

In [ ]:
# -------------------------------------------------------------
# PART 4: Task 4.2 Hyperparameter Optimization Sweeps
# -------------------------------------------------------------

import pandas as pd
import torch

print("--- Compiling Hyperparameter Sensitivity Log ---")

learning_rates_to_test = [
    1e-3,
    3e-4,
    5e-5
]

results_grid = []

# Clear old CUDA cache before experiments
torch.cuda.empty_cache()

for test_lr in learning_rates_to_test:

    print(f"\nTesting Learning Rate: {test_lr}")

    sweep_config = TransformerConfig(

        vocabulary_size=50257,

        context_window=256,

        num_layers=2,

        num_heads=2,

        embedding_dim=128,

        dropout_rate=0.1
    )

    sweep_model = NanoTransformer(
        sweep_config
    ).to(device)

    # Save original LR
    old_lr = SETTINGS["learning_rate"]

    # Update LR temporarily
    SETTINGS["learning_rate"] = test_lr

    # Train experiment model
    sweep_history = train_model(

        sweep_model,

        dataset_name="poems",

        training_steps=150,

        evaluation_interval=50
    )

    # Save results
    results_grid.append({

        "Learning Rate": test_lr,

        "Context Window": 256,

        "Final Training Loss":
            sweep_history["train_loss"][-1],

        "Final Validation Loss":
            sweep_history["validation_loss"][-1]
    })

    # Restore original LR
    SETTINGS["learning_rate"] = old_lr

    # Clear CUDA cache after each experiment
    torch.cuda.empty_cache()

# Create results table
df_sensitivity = pd.DataFrame(results_grid)

# Display table
print("\nHyperparameter Sensitivity Results:\n")

print(df_sensitivity)

# Optional markdown display
print("\nMarkdown Table:\n")

print(df_sensitivity.to_markdown(index=False))

In [ ]:
tiny_config = TransformerConfig(

    context_window=256,
    vocabulary_size=50257,

    num_layers=2,
    num_heads=2,
    embedding_dim=64,

    dropout_rate=0.1
)

In [ ]:
medium_config = TransformerConfig(

    context_window=256,
    vocabulary_size=50257,

    num_layers=4,
    num_heads=4,
    embedding_dim=128,

    dropout_rate=0.1
)

In [ ]:
large_config = TransformerConfig(

    context_window=256,
    vocabulary_size=50257,

    num_layers=8,
    num_heads=8,
    embedding_dim=256,

    dropout_rate=0.1
)

In [ ]:
no_position_config = TransformerConfig(

    use_positional_embeddings=False
)

no_position_model = NanoTransformer(
    no_position_config
).to(DEVICE)

In [ ]:
no_mask_config = TransformerConfig(

    use_causal_mask=False
)

no_mask_model = NanoTransformer(
    no_mask_config
).to(DEVICE)

In [ ]:
no_residual_config = TransformerConfig(

    use_residual_connections=False
)

no_residual_model = NanoTransformer(
    no_residual_config
).to(DEVICE)

In [ ]:
no_layernorm_config = TransformerConfig(

    use_layernorm=False
)

no_layernorm_model = NanoTransformer(
    no_layernorm_config
).to(DEVICE)

In [ ]:
learning_rates = [
    1e-4,
    3e-4,
    1e-3
]

lr_results = {}

for lr in learning_rates:

    print(f"\nTesting LR = {lr}")

    SETTINGS["learning_rate"] = lr

    config = TransformerConfig(

        vocabulary_size=50257,

        context_window=256,

        num_layers=2,

        num_heads=2,

        embedding_dim=64,

        dropout_rate=0.1
    )

    model = NanoTransformer(config).to(
        device
    )

    history = train_model(

        model,

        dataset_name="poems",

        training_steps=200,

        evaluation_interval=50
    )

    lr_results[lr] = (
        history["validation_loss"][-1]
    )

In [ ]:
# -------------------------------------------------------------
# PART 4: Task 4.3 Scaling Experiment Logs
# -------------------------------------------------------------
print("--- Verifying Empirical Transformer Scaling Laws ---")

scaling_profiles = {
    "Tiny (2 Layers, 128 Dim)": TransformerConfig(vocabulary_size=50257, context_window=256, num_layers=2, num_heads=2, embedding_dim=128),
    "Medium (4 Layers, 256 Dim)": TransformerConfig(vocabulary_size=50257, context_window=256, num_layers=4, num_heads=4, embedding_dim=256),
    "Larger (6 Layers, 384 Dim)": TransformerConfig(vocabulary_size=50257, context_window=256, num_layers=6, num_heads=6, embedding_dim=384)
}

for label, profile_cfg in scaling_profiles.items():
    scale_model = NanoTransformer(profile_cfg).to(DEVICE)
    print(f"Initialized Config Profile: {label} | Total Parameters: {scale_model.count_parameters()/1e6:.2f}M")
    # Run a short tracking loop (e.g., 100 steps) to record cross-entropy improvement speeds
    scale_history = train_model(scale_model, dataset_name=label, training_steps=100, evaluation_interval=100)
    print(f"==> {label} Performance - Final Validation Loss: {scale_history['validation_loss'][-1]:.4f}\n")